# 4-1절 연습 문제 풀이

이 노트북은 4-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch04/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 3장 회오리 데이터 - 4장 연습에서 공통으로 사용한다.
import csv

def load_spiral(path='../../data/ch3_spiral_data.csv'):
    with open(path, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    X = torch.tensor([[float(r['x1']), float(r['x2'])] for r in rows])
    Y = torch.tensor([int(float(r['label'])) for r in rows])
    return X, Y

def split_data(X, Y, ratios=(0.6, 0.2, 0.2), seed=SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(X), generator=g)
    n_train = int(len(X) * ratios[0])
    n_valid = int(len(X) * ratios[1])
    parts = [idx[:n_train], idx[n_train:n_train + n_valid], idx[n_train + n_valid:]]
    return [(X[p], Y[p]) for p in parts]

## 연습 4-1

조기 종료 방식으로 모델을 학습하는 [코드 4-3] 예제에서 참을성 한계를 늘리거나 줄이면 모델의 학습 과정이 어떻게 바뀔지 예측한 후, 직접 값을 바꿔 가며 실제 결과가 예측과 일치하는지 확인해 보자.

In [ ]:
X, Y = load_spiral()
(Xtr, Ytr), (Xva, Yva), (Xte, Yte) = split_data(X, Y)

def train_early_stopping(patience, epochs=2000, lr=0.05, verbose=False):
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                          nn.Linear(32, 32), nn.ReLU(),
                          nn.Linear(32, int(Y.max()) + 1))
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_epoch, wait, best_state = float('inf'), 0, 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        loss = criterion(model(Xtr), Ytr)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(Xva), Yva).item()
        if val_loss < best_loss:
            best_loss, best_epoch, wait = val_loss, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    with torch.no_grad():
        acc = ((model(Xte).argmax(dim=1) == Yte).float().mean() * 100).item()
    return best_epoch, epoch, best_loss, acc

print(f'{"참을성":>6} {"최적 에포크":>10} {"종료 에포크":>10} {"검증 손실":>10} {"평가 정확도":>10}')
for patience in (5, 20, 100, 300):
    b, e, l, a = train_early_stopping(patience)
    print(f'{patience:6d} {b:10d} {e:10d} {l:10.4f} {a:9.2f}%')

**예상**: 참을성이 작으면 일시적인 정체에도 학습이 멈춰 **과소적합**되기 쉽고, 크면 과적합 구간까지 오래 학습하지만 최적 파라미터는 보존되므로 최종 성능은 비슷하되 **학습 시간이 늘어난다**.

실제 결과도 같은 경향을 보인다. 참을성은 '검증 손실이 우연히 나빠지는 구간을 얼마나 참아 줄 것인가'를 정하는 값이므로, 손실이 출렁이는 문제일수록 크게 잡아야 한다.

## 연습 4-2

데이터를 훈련, 검증, 평가 데이터셋으로 구분할 때의 비율은 상황에 따라 다르지만 훈련 데이터셋을 최대한 확보한 후 검증과 평가 데이터셋의 비율을 정하는 것이 일반적이다. 훈련 데이터셋을 최대한 확보하는 이유가 무엇인지 설명해 보자. 한편 검증 데이터셋의 양도 무조건 많다고 좋은 것은 아니다. 검증 데이터셋의 양이 과도할 때 생길 수 있는 문제가 무엇일지 유추해 보자. 만약 감이 잘 잡히지 않는다면, 다양한 비율로 나눈 데이터를 사용해 모델을 반복 학습해 보자.

### 풀이

**훈련 데이터셋을 최대한 확보하는 이유**
모델이 학습할 수 있는 정보의 양이 훈련 데이터의 양에 비례하기 때문이다. 데이터가 적으면 모델이 일반적인 규칙 대신 몇몇 샘플을 외워 버려(과적합) 처음 보는 데이터에서 성능이 떨어진다. 검증·평가 데이터는 **측정에만** 쓰이고 모델의 능력을 직접 키우지는 않는다.

**검증 데이터셋이 과도하게 많을 때의 문제**
1. 훈련 데이터가 그만큼 줄어 모델 성능 자체가 떨어진다.
2. 매 에포크 검증을 수행하므로 학습 시간이 길어진다.
3. 검증 지표의 정밀도는 어느 수준을 넘으면 거의 나아지지 않는다. 즉 **얻는 것에 비해 잃는 것이 크다**.

정리하면 검증 데이터는 '조기 종료 시점이나 하이퍼파라미터를 안정적으로 고를 수 있을 만큼'만 있으면 충분하고, 나머지는 훈련에 쓰는 편이 낫다.

## 연습 4-3

3-3절의 회오리 모양 데이터 분류 모델 중 평균제곱오차를 사용해 학습한 모델과, 교차 엔트로피 손실 함수에 은닉층 시그모이드 활성화를 사용한 모델 각각에 조기 종료 방식을 적용해 학습해 보자. 최적의 모델을 찾아 정확도를 확인하고 [코드 4-3]의 결과와 비교해 보자.

힌트: 두 모델과 [코드 4-3]은 모두 검증 손실이 정체되는 형태가 다를 수 있다. 최적의 모델을 찾으려면 단순히 조기 종료 방식만 적용해서는 안 되고 다른 하이퍼파라미터의 수정이 필요할 수도 있다.

In [ ]:
def train_variant(kind, patience=100, epochs=3000, lr=0.05):
    torch.manual_seed(SEED)
    n_class = int(Y.max()) + 1
    if kind == 'MSE':
        model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                              nn.Linear(32, 32), nn.ReLU(),
                              nn.Linear(32, n_class), nn.Softmax(dim=1))
        criterion = nn.MSELoss()
        to_target = lambda y: nn.functional.one_hot(y, n_class).float()
    else:   # 교차 엔트로피 + 은닉층 시그모이드
        model = nn.Sequential(nn.Linear(2, 32), nn.Sigmoid(),
                              nn.Linear(32, 32), nn.Sigmoid(),
                              nn.Linear(32, n_class))
        criterion = nn.CrossEntropyLoss()
        to_target = lambda y: y
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best, wait, best_state, best_epoch = float('inf'), 0, None, 0
    for epoch in range(1, epochs + 1):
        loss = criterion(model(Xtr), to_target(Ytr))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        with torch.no_grad():
            vl = criterion(model(Xva), to_target(Yva)).item()
        if vl < best:
            best, wait, best_epoch = vl, 0, epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    with torch.no_grad():
        acc = ((model(Xte).argmax(dim=1) == Yte).float().mean() * 100).item()
    return best_epoch, acc

for kind in ('MSE', 'CE+Sigmoid'):
    ep, acc = train_variant(kind)
    print(f'{kind:12s} 최적 에포크 {ep:5d}, 평가 정확도 {acc:.2f}%')

힌트대로 두 모델은 검증 손실이 정체되는 모습이 다르다.

- **평균제곱오차 + 소프트맥스**: 출력이 이미 확률이라 오차가 작게 계산되고 기울기도 작아 수렴이 느리다. 참을성을 넉넉히 주어야 최적점을 지나치지 않는다.
- **교차 엔트로피 + 시그모이드 은닉층**: 시그모이드 때문에 기울기가 작아져 초반 학습이 더디다.

조기 종료만 적용하면 두 모델 모두 일찍 멈출 수 있으므로, **참을성과 학습률을 함께 조정**해야 [코드 4-3]과 견줄 성능이 나온다.

## 연습 4-4

[도전 문제] [코드 4-3] 예제에서는 검증 손실을 조기 종료 여부를 따지는 지표로 사용한다. 그런데 분류 모델에서는 손실만큼이나 분류 정확도도 중요한 검증 지표로 활용할 수 있다. 분류 정확도를 검증 지표로 사용하도록 [코드 4-3] 예제를 수정한 후, 결과를 확인해 보자.

In [ ]:
# 검증 지표를 손실 대신 분류 정확도로 바꾼다.
def train_by_metric(metric='loss', patience=100, epochs=3000, lr=0.05):
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                          nn.Linear(32, 32), nn.ReLU(),
                          nn.Linear(32, int(Y.max()) + 1))
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # 손실은 작을수록, 정확도는 클수록 좋으므로 비교 방향이 반대다.
    best = float('inf') if metric == 'loss' else -1.0
    wait, best_state, best_epoch = 0, None, 0
    for epoch in range(1, epochs + 1):
        loss = criterion(model(Xtr), Ytr)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        with torch.no_grad():
            out = model(Xva)
            score = (criterion(out, Yva).item() if metric == 'loss'
                     else (out.argmax(dim=1) == Yva).float().mean().item())
        improved = score < best if metric == 'loss' else score > best
        if improved:
            best, wait, best_epoch = score, 0, epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    with torch.no_grad():
        acc = ((model(Xte).argmax(dim=1) == Yte).float().mean() * 100).item()
    return best_epoch, best, acc

for metric in ('loss', 'accuracy'):
    ep, best, acc = train_by_metric(metric)
    print(f'{metric:9s} 기준 - 최적 에포크 {ep:5d}, 검증 지표 {best:.4f}, 평가 정확도 {acc:.2f}%')

정확도를 지표로 쓸 때 주의할 점이 두 가지다.

1. **비교 방향이 반대**다. 손실은 작아져야 개선이고 정확도는 커져야 개선이다.
2. 정확도는 **계단식으로 변하는 이산적인 값**이라, 손실은 계속 나아지는데 정확도는 그대로인 구간이 생긴다. 그래서 정확도만 보면 조기 종료가 일찍 걸릴 수 있다.

실무에서는 둘을 함께 기록하고, 최종 목적(정확도 최대화 등)에 맞는 지표를 조기 종료 기준으로 삼는다.